In [8]:
import os
import torch
import monai
from monai.transforms import (
    LoadImaged, AddChanneld, ScaleIntensityd, Resized, EnsureTyped, Compose
)
from monai.data import Dataset, DataLoader, CacheDataset
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.transforms import AsDiscreted
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

ImportError: cannot import name 'AddChanneld' from 'monai.transforms' (/user1/ngmm/tr855969/.conda/envs/env_tf/lib/python3.10/site-packages/monai/transforms/__init__.py)

In [7]:
# Step 1: Set up the environment
!pip install monai


In [ ]:
# Define paths
data_dir = "path_to_your_data_directory"
image_paths = sorted(glob(os.path.join(data_dir, "images", "*.nii.gz")))
label_paths = sorted(glob(os.path.join(data_dir, "labels", "*.nii.gz")))

# Step 2: Define data transforms
train_transforms = Compose([
    LoadImage(image_only=True),
    AddChannel(),
    ScaleIntensity(),
    Resize((128, 128, 128)),  # Adjust size as needed
    EnsureType()
])

val_transforms = Compose([
    LoadImage(image_only=True),
    AddChannel(),
    ScaleIntensity(),
    Resize((128, 128, 128)),  # Adjust size as needed
    EnsureType()
])

# Create a dataset and data loader
train_files = [{"image": img, "label": lbl} for img, lbl in zip(image_paths, label_paths)]
val_files = [{"image": img, "label": lbl} for img, lbl in zip(image_paths, label_paths)]

train_ds = CacheDataset(data=train_files, transform=train_transforms, cache_rate=1.0)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)

val_ds = CacheDataset(data=val_files, transform=val_transforms, cache_rate=1.0)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

# Step 3: Define the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(
    dimensions=3,
    in_channels=1,
    out_channels=24,  # 24 regions to segment
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

# Define loss function and optimizer
loss_function = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Define metric for evaluation
dice_metric = DiceMetric(include_background=True, reduction="mean", get_not_nans=False)

# Step 4: Training loop
max_epochs = 100
val_interval = 2

for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        print(f"{step}/{len(train_ds) // train_loader.batch_size}, train_loss: {loss.item():.4f}")

    epoch_loss /= step
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
                val_outputs = sliding_window_inference(val_inputs, (128, 128, 128), 4, model)
                val_outputs = torch.argmax(val_outputs, dim=1).detach().cpu()
                val_labels = val_labels.cpu()
                dice_metric(y_pred=val_outputs, y=val_labels)

            metric = dice_metric.aggregate().item()
            dice_metric.reset()
            print(f"epoch {epoch + 1} validation dice: {metric:.4f}")

# Save the trained model
torch.save(model.state_dict(), "mouse_brain_segmentation_model.pth")

# Step 5: Post-process and visualize the results
# Load the model for inference
model.load_state_dict(torch.load("mouse_brain_segmentation_model.pth"))
model.eval()

# Perform inference on a sample image
sample_image_path = "path_to_sample_image.nii.gz"
sample_image = val_transforms(sample_image_path)["image"].to(device)

with torch.no_grad():
    sample_output = sliding_window_inference(sample_image, (128, 128, 128), 4, model)
    sample_output = torch.argmax(sample_output, dim=1).detach().cpu().numpy()

# Visualize the results
for i in range(0, sample_output.shape[1], sample_output.shape[1] // 10):
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(sample_image.cpu().numpy()[0, i, :, :], cmap="gray")
    plt.subplot(1, 2, 2)
    plt.title("Segmented Image")
    plt.imshow(sample_output[0, i, :, :], cmap="nipy_spectral")
    plt.show()
